In [26]:
!pip3 install -q --upgrade pip
!pip3 install -q pandas matplotlib seaborn openpyxl

In [27]:
import os
import pandas as pd
from itertools import product

In [28]:
# Define the bounding box for The Gambia
GAMBIA_BOUNDARIES = {
    "min_lat": 13.052113101601,  # Lower latitude
    "max_lat": 13.837920583180,  # Upper latitude
    "min_lon": -17.06499856438,  # More western longitude
    "max_lon": -13.77921401885   # More eastern longitude
}


def filter_gambia_points(file_path: str) -> pd.DataFrame:
    """
    Read the ASCII file and filter points that are within The Gambia's boundaries.
    """
    df = pd.read_csv(file_path, delim_whitespace=True)
    # Filter based on The Gambia's bounding box
    df = df[(df['X'] >= GAMBIA_BOUNDARIES['min_lon']) & (df['X'] <= GAMBIA_BOUNDARIES['max_lon']) &
            (df['Y'] >= GAMBIA_BOUNDARIES['min_lat']) & (df['Y'] <= GAMBIA_BOUNDARIES['max_lat'])]
    return df

def map_values_to_description(df: pd.DataFrame, value_key: str, mapping_dict: dict) -> pd.DataFrame:
    """
    Maps the ASCII values in the DataFrame to their descriptions using the provided mapping dictionary.
    """
    df[value_key] = df[value_key].map(mapping_dict)
    return df

def round_coordinates(df, precision=3):
    """
    Round the X and Y coordinates to the specified precision.
    
    Parameters:
    df (pd.DataFrame): DataFrame with the X and Y columns.
    precision (int): Decimal places to round to.

    Returns:
    pd.DataFrame: DataFrame with rounded coordinates.
    """
    df['X'] = df['X'].round(precision)
    df['Y'] = df['Y'].round(precision)
    return df

# Depth to Groundwater mapping
depth_to_groundwater_mapping = {
    'VS': '0-7', 'S': '7-25', 'SM': '25-50', 'M': '50-100', 'D': '100-250', 'VD': '>250'
}

# Groundwater Productivity mapping
groundwater_productivity_mapping = {
    'VH': '>20', 'H': '5-20', 'M': '1-5', 'LM': '0.5-1', 'L': '0.1-0.5', 'VL': '<0.1'
}

# Groundwater Storage mapping
groundwater_storage_mapping = {
    '0': '0', 'L': '<1000', 'LM': '1000-10,000', 'M': '10,000-25,000', 'H': '25,000-50,000', 'VH': '>50,000'
}

# Paths to the data files
depth_to_groundwater_file = 'original_data/DepthToGroundwater_V2/xyzASCII_dtwmap_v2.txt'
groundwater_productivity_file = 'original_data/GroundwaterProductivity/xyzASCII_gwprod_v1.txt'
groundwater_storage_file = 'original_data/GroundwaterStorage/xyzASCII_gwstor_v1.txt'


# Read and process the Depth to Groundwater data
df_dtw = filter_gambia_points(file_path=depth_to_groundwater_file)
df_dtw = map_values_to_description(df=df_dtw, value_key='DTWAFRICA_', mapping_dict=depth_to_groundwater_mapping)
df_dtw.rename(columns={'DTWAFRICA_': 'DepthToGroundwater'}, inplace=True)

# Read and process the Groundwater Productivity data
df_gwprod = filter_gambia_points(file_path=groundwater_productivity_file)
df_gwprod = map_values_to_description(df=df_gwprod, value_key='GWPROD_V2', mapping_dict=groundwater_productivity_mapping)
df_gwprod.rename(columns={'GWPROD_V2': 'GroundwaterProductivity'}, inplace=True)

# Read and process the Groundwater Storage data
df_gwstor = filter_gambia_points(file_path=groundwater_storage_file)
df_gwstor = map_values_to_description(df=df_gwstor, value_key='GWSTOR_V2', mapping_dict=groundwater_storage_mapping)
df_gwstor.rename(columns={'GWSTOR_V2': 'GroundwaterStorage'}, inplace=True)


# Apply the rounding to each DataFrame before merging
precision = 2
df_dtw = round_coordinates(df_dtw, precision=precision)
df_gwprod = round_coordinates(df_gwprod, precision=precision)
df_gwstor = round_coordinates(df_gwstor, precision=precision)

# Define a function to get the most frequent value, or return a custom value if there's a tie
def most_frequent(series):
    counts = series.value_counts()
    if len(counts) == 0:
        return None
    elif len(counts) > 1 and counts.iloc[0] == counts.iloc[1]:
        # Handle the tie case here if necessary, for example by returning a specific value
        return 'Tie' 
    return counts.idxmax()


# Now combine the dataframes
combined_df = df_dtw.merge(df_gwprod, on=['X', 'Y'], how='outer').merge(df_gwstor, on=['X', 'Y'], how='outer')

# Group by the rounded coordinates and aggregate using the most frequent value
aggregated_df = combined_df.groupby(['X', 'Y']).agg({
    'DepthToGroundwater': most_frequent,
    'GroundwaterProductivity': most_frequent,
    'GroundwaterStorage': most_frequent
}).reset_index()

# Define the path to the folder where you want to save the CSV file
folder_path = 'processed_data/'

# Ensure the folder exists, if not, create it
if not os.path.exists(folder_path):
    os.makedirs(folder_path)

# Define the name of the CSV file
csv_file_name = 'combined_data.csv'

# Construct the full path to the CSV file
combined_csv_path = folder_path + csv_file_name

# Save the combined data to the CSV file
combined_df.to_csv(combined_csv_path, index=False)

print(f"Combined data saved to {combined_csv_path}")


Combined data saved to processed_data/combined_data.csv


In [32]:
# Load your data
df = pd.read_csv('processed_data/combined_data.csv')

# Define the rounding precision based on the desired range for grouping
# For example, 0.03 would group points that are within ~3km of each other
grouping_threshold = 0.01

# Round the coordinates to the nearest grouping threshold
df['X_rounded'] = (df['X'] / grouping_threshold).round() * grouping_threshold
df['Y_rounded'] = (df['Y'] / grouping_threshold).round() * grouping_threshold

# Now group by these rounded coordinates
grouped = df.groupby(['X_rounded', 'Y_rounded'])

# Define an aggregation function that takes the most frequent non-null value
def most_frequent_non_null(series):
    if series.isnull().all():
        return None
    else:
        return series.dropna().mode().iloc[0]

# Aggregate your data
aggregated_df = grouped.agg({
    'X': 'first',  # Keep the first X value
    'Y': 'first',  # Keep the first Y value
    'DepthToGroundwater': most_frequent_non_null,
    'GroundwaterProductivity': most_frequent_non_null,
    'GroundwaterStorage': most_frequent_non_null
}).reset_index(drop=True)

# Save the aggregated data back to CSV
aggregated_df.to_csv('processed_data/combined_data_aggregated.csv', index=False)

print(f"Aggregated data saved to combined_data_aggregated.csv")


Aggregated data saved to combined_data_aggregated.csv


In [33]:
# Load the aggregated data
df = pd.read_csv('processed_data/combined_data_aggregated.csv')

# Calculate the total number of rows
total_rows = len(df)

# Calculate the number of rows with all values present
full_values = df.dropna().shape[0]

# Calculate the number of rows with missing values
missing_values = total_rows - full_values

# Calculate the number of rows with only the first value present
only_first = df[df['GroundwaterProductivity'].isna() & df['GroundwaterStorage'].isna()].shape[0]

# Calculate the number of rows with only the second value present
only_second = df[df['DepthToGroundwater'].isna() & df['GroundwaterStorage'].isna()].shape[0]

# Calculate the number of rows with only the third value present
only_third = df[df['DepthToGroundwater'].isna() & df['GroundwaterProductivity'].isna()].shape[0]

# Calculate the number of rows with the first and second values present but missing third
first_and_second = df[df['GroundwaterStorage'].isna() & df['DepthToGroundwater'].notna() & df['GroundwaterProductivity'].notna()].shape[0]

# Calculate the number of rows with the second and third values present but missing first
second_and_third = df[df['DepthToGroundwater'].isna() & df['GroundwaterProductivity'].notna() & df['GroundwaterStorage'].notna()].shape[0]

# Calculate the number of rows with the first and third values present but missing second
first_and_third = df[df['GroundwaterProductivity'].isna() & df['DepthToGroundwater'].notna() & df['GroundwaterStorage'].notna()].shape[0]

# Print the results
print(f"Total rows: {total_rows}")
print(f"Rows with all values: {full_values}")
print(f"Rows with any missing values: {missing_values}")
print(f"Rows with only the first value: {only_first}")
print(f"Rows with only the second value: {only_second}")
print(f"Rows with only the third value: {only_third}")
print(f"Rows with the first and second values, but missing third: {first_and_second}")
print(f"Rows with the second and third values, but missing first: {second_and_third}")
print(f"Rows with the first and third values, but missing second: {first_and_third}")


Total rows: 2671
Rows with all values: 0
Rows with any missing values: 2671
Rows with only the first value: 933
Rows with only the second value: 826
Rows with only the third value: 912
Rows with the first and second values, but missing third: 0
Rows with the second and third values, but missing first: 0
Rows with the first and third values, but missing second: 0


In [35]:
import pandas as pd
from sklearn.neighbors import NearestNeighbors

# Load the data
df = pd.read_csv('processed_data/combined_data_aggregated.csv')

# Fill in missing values with empty string for easier processing
df.fillna('', inplace=True)

# Initialize NearestNeighbors
neigh = NearestNeighbors(n_neighbors=5)

# Fit the model on the available X and Y coordinates
neigh.fit(df[['X', 'Y']])

# Define a function to impute missing values for a given column
def impute_column(df, column_name):
    # Work with the subset of data that needs imputation for this column
    needs_impute = df[df[column_name] == '']
    for index, row in needs_impute.iterrows():
        # Find the indices of the closest neighbors
        _, indices = neigh.kneighbors([row[['X', 'Y']].to_numpy()])
        # Retrieve the neighbors' values for the column to impute
        neighbor_values = df.iloc[indices[0]][column_name].values
        # Filter out empty values
        neighbor_values = [value for value in neighbor_values if value != '']
        if neighbor_values:
            # Choose the most common non-empty value among the neighbors
            most_common_value = pd.Series(neighbor_values).mode().iloc[0]
            # Impute the missing value
            df.at[index, column_name] = most_common_value

# Impute 'GroundwaterProductivity'
impute_column(df, 'GroundwaterProductivity')

# Impute 'GroundwaterStorage'
impute_column(df, 'GroundwaterStorage')

# Replace empty strings back to NaN to maintain consistency
df.replace('', pd.NA, inplace=True)

# Save the filled in data back to CSV
df.to_csv('processed_data/combined_data_filled.csv', index=False)


/Users/franciscofurey/00DataScience/OpenAi/venv/lib/python3.9/site-packages/sklearn/base.py:464: UserWarning: X does not have valid feature names, but NearestNeighbors was fitted with feature names
  warnings.warn(
/Users/franciscofurey/00DataScience/OpenAi/venv/lib/python3.9/site-packages/sklearn/base.py:464: UserWarning: X does not have valid feature names, but NearestNeighbors was fitted with feature names
  warnings.warn(
/Users/franciscofurey/00DataScience/OpenAi/venv/lib/python3.9/site-packages/sklearn/base.py:464: UserWarning: X does not have valid feature names, but NearestNeighbors was fitted with feature names
  warnings.warn(
/Users/franciscofurey/00DataScience/OpenAi/venv/lib/python3.9/site-packages/sklearn/base.py:464: UserWarning: X does not have valid feature names, but NearestNeighbors was fitted with feature names
  warnings.warn(
/Users/franciscofurey/00DataScience/OpenAi/venv/lib/python3.9/site-packages/sklearn/base.py:464: UserWarning: X does not have valid feature 

In [37]:
# Load the aggregated data
df = pd.read_csv('processed_data/combined_data_filled.csv')

# Calculate the total number of rows
total_rows = len(df)

# Calculate the number of rows with all values present
full_values = df.dropna().shape[0]

# Calculate the number of rows with missing values
missing_values = total_rows - full_values

# Calculate the number of rows with only the first value present
only_first = df[df['GroundwaterProductivity'].isna() & df['GroundwaterStorage'].isna()].shape[0]

# Calculate the number of rows with only the second value present
only_second = df[df['DepthToGroundwater'].isna() & df['GroundwaterStorage'].isna()].shape[0]

# Calculate the number of rows with only the third value present
only_third = df[df['DepthToGroundwater'].isna() & df['GroundwaterProductivity'].isna()].shape[0]

# Calculate the number of rows with the first and second values present but missing third
first_and_second = df[df['GroundwaterStorage'].isna() & df['DepthToGroundwater'].notna() & df['GroundwaterProductivity'].notna()].shape[0]

# Calculate the number of rows with the second and third values present but missing first
second_and_third = df[df['DepthToGroundwater'].isna() & df['GroundwaterProductivity'].notna() & df['GroundwaterStorage'].notna()].shape[0]

# Calculate the number of rows with the first and third values present but missing second
first_and_third = df[df['GroundwaterProductivity'].isna() & df['DepthToGroundwater'].notna() & df['GroundwaterStorage'].notna()].shape[0]

# Print the results
print(f"Total rows: {total_rows}")
print(f"Rows with all values: {full_values}")
print(f"Rows with any missing values: {missing_values}")
print(f"Rows with only the first value: {only_first}")
print(f"Rows with only the second value: {only_second}")
print(f"Rows with only the third value: {only_third}")
print(f"Rows with the first and second values, but missing third: {first_and_second}")
print(f"Rows with the second and third values, but missing first: {second_and_third}")
print(f"Rows with the first and third values, but missing second: {first_and_third}")

Total rows: 2671
Rows with all values: 916
Rows with any missing values: 1755
Rows with only the first value: 14
Rows with only the second value: 0
Rows with only the third value: 4
Rows with the first and second values, but missing third: 0
Rows with the second and third values, but missing first: 1734
Rows with the first and third values, but missing second: 3
